# Exploration et Profiling des données IRVE

Ce notebook sert à analyser notre jeu de données brut avant de concevoir la base MongoDB.
On mesure les volumes réels, la cardinalité entre les stations et les points de recharge, la qualité des coordonnées GPS et la couverture des données dynamiques pour justifier nos choix de schéma.

In [1]:
import os
import pandas as pd
import numpy as np

DATA_DIR = os.path.join("..", "data")
CSV_STATIQUE = os.path.join(DATA_DIR, "consolidation_transport_irve_statique.csv")
CSV_DYNAMIQUE = os.path.join(DATA_DIR, "consolidation-nationale-irve-dynamique-2026-08-27T14_31_23.844081Z.csv")

## 1. Analyse du fichier statique (Stations et Points de recharge)

In [2]:
df_s = pd.read_csv(CSV_STATIQUE, sep=None, engine='python')

id_station_col = [c for c in df_s.columns if 'id_station' in c][0]
id_pdc_col = [c for c in df_s.columns if 'id_pdc' in c][0]
lon_col = [c for c in df_s.columns if 'longitude' in c][0]
lat_col = [c for c in df_s.columns if 'latitude' in c][0]

nb_stations = df_s[id_station_col].nunique()
nb_pdc_uniques = df_s[id_pdc_col].nunique()
pdc_per_st = df_s.groupby(id_station_col)[id_pdc_col].nunique()
valides_geo = df_s[(df_s[lon_col].notna()) & (df_s[lat_col].notna()) & (df_s[lon_col] != 0) & (df_s[lat_col] != 0)]
flag_geo_ok = (df_s['consolidated_is_lon_lat_correct'] == True).sum()
flag_geo_warn = (df_s['consolidated_is_lon_lat_correct'] == False).sum()
nb_geo_manquantes = len(df_s) - len(valides_geo)

print(f"Total points de charge (lignes) : {len(df_s):,}")
print(f"Nombre de stations uniques      : {nb_stations:,}")
print(f"Nombre de PDC uniques           : {nb_pdc_uniques:,}")
print("Distribution des PDC uniques par station :")
print(f"  - Moyenne : {pdc_per_st.mean():.2f} | Médiane (p50) : {int(pdc_per_st.median())} | p75 : {int(pdc_per_st.quantile(0.75))} | p95 : {int(pdc_per_st.quantile(0.95))} | p99 : {int(pdc_per_st.quantile(0.99))} | Max : {pdc_per_st.max()}")
print("Coordonnées GPS :")
print(f"  - Renseignées et non nulles   : {len(valides_geo):,} ({len(valides_geo)/len(df_s)*100:.2f}% - {nb_geo_manquantes} manquantes/nulles)")
print(f"  - Validées par la source      : {flag_geo_ok:,} ({flag_geo_ok/len(df_s)*100:.1f}%)")
print(f"  - À contrôler / nettoyer      : {flag_geo_warn:,} ({flag_geo_warn/len(df_s)*100:.1f}%)")


Total points de charge (lignes) : 165,595
Nombre de stations uniques      : 48,040
Nombre de PDC uniques           : 165,593
Distribution des PDC uniques par station :
  - Moyenne : 3.45 | Médiane (p50) : 2 | p75 : 4 | p95 : 10 | p99 : 20 | Max : 505
Coordonnées GPS :
  - Renseignées et non nulles   : 165,584 (99.99% - 11 manquantes/nulles)
  - Validées par la source      : 139,248 (84.1%)
  - À contrôler / nettoyer      : 26,347 (15.9%)


### Justification : Station -> Points de recharge

On a d'abord mesuré la cardinalité réelle avant de choisir la modélisation. Le dataset contient 165 595 lignes pour 48 040 stations uniques, avec 165 593 points de recharge distincts. Une station possède en moyenne 3,45 PDC, avec une médiane de 2. 75 % des stations ont 4 PDC ou moins, 95 % en ont 10 ou moins et 99 % en ont 20 ou moins.

On a donc choisi d'embarquer les points de recharge dans un tableau `points_recharge` au sein du document station, vu que la cardinalité reste très faible dans la grande majorité des cas et que les bornes appartiennent naturellement à leur station.

Le coût de ce choix, c'est que les requêtes et agrégations au niveau du point de recharge demandent souvent un `$unwind`, ce qui augmente le nombre de documents intermédiaires en mémoire. On a aussi quelques stations atypiques beaucoup plus grosses, avec un maximum observé de 505 PDC.

## 2. Analyse des données dynamiques (Statuts temps réel)

In [3]:
df_d = pd.read_csv(CSV_DYNAMIQUE, sep=None, engine='python')
id_pdc_dyn_col = [c for c in df_d.columns if 'id_pdc' in c][0]
nb_pdc_dyn = df_d[id_pdc_dyn_col].nunique()
pdc_match = set(df_s[id_pdc_col]).intersection(set(df_d[id_pdc_dyn_col]))
dyn_par_pdc = df_d.groupby(id_pdc_dyn_col).size()

print(f"Total observations dynamiques (snapshot) : {len(df_d):,}")
print(f"PDC uniques avec statut                  : {nb_pdc_dyn:,}")
print(f"PDC statiques avec statut dynamique      : {len(pdc_match):,} ({len(pdc_match)/df_s[id_pdc_col].nunique()*100:.1f}%)")
print(f"PDC présents plusieurs fois (>1)         : {(dyn_par_pdc > 1).sum():,} (max: {dyn_par_pdc.max()} lignes/PDC)")

Total observations dynamiques (snapshot) : 115,159
PDC uniques avec statut                  : 104,046
PDC statiques avec statut dynamique      : 100,838 (60.9%)
PDC présents plusieurs fois (>1)         : 11,098 (max: 4 lignes/PDC)


### Justification : Point de recharge -> Statut dynamique

Le fichier dynamique contient 115 159 lignes pour 104 046 PDC uniques. Parmi les PDC du fichier statique, 100 838 ont un statut dynamique, soit environ 60,9 % de couverture.

On a choisi de séparer ces données dans une collection `statuts_pdc` distincte pour découpler les mises à jour fréquentes d'occupation et d'état de fonctionnement des caractéristiques statiques de la station, vu qu'elles n'ont pas du tout le même cycle de vie. Ça évite de réécrire le document station à chaque changement d'état.

Le coût de ce choix, c'est que les requêtes croisant les caractéristiques techniques d'un PDC et sa disponibilité en direct demanderont une jointure `$lookup` ou plusieurs lectures.

## 3. Qualité des coordonnées géographiques pour l'index 2dsphere

Sur les 165 595 lignes, 165 584 ont des coordonnées renseignées et non nulles (99,99 %, il n'y a que 11 lignes vides ou à zéro comme des stations de test).

Par contre, le champ `consolidated_is_lon_lat_correct` indique que seulement 84,1 % sont explicitement validées par la source, et 26 347 lignes (15,9 %) sont marquées à vérifier.

On prévoit donc une étape de nettoyage lors de l'import pour s'assurer que seules les coordonnées valides en France soient transformées en GeoJSON `Point [longitude, latitude]` pour alimenter notre index `2dsphere`.